# CSCIE89 : Timeseries Prediction using RNN 

We will train a small RNN neural network from scratch on timeseries data  We use the previous **60 trading observations** of Micron Technology's (MU) opening price to predict the next opening price.

The architecture has four RNN layers with 50 hidden units each, dropout of 0.2 after each layer, and a linear output producing one value. Data preparation, the 100-epoch training schedule, and rolling one-step evaluation follow the earlier notebook. Gradient clipping is added to help control exploding gradients in the vanilla RNN.


## 1. Dependencies and configuration
Select a Python kernel with PyTorch, NumPy, pandas, matplotlib, and scikit-learn installed. If needed, uncomment the install command below and restart the kernel afterward.

Set `DATA_DIR` to the folder containing your original CSVs. 

In [ ]:
# %pip install torch numpy pandas matplotlib scikit-learn

In [ ]:
# Step 1: Import libraries, choose hyperparameters, seed randomness, and select a device.
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

DATA_DIR = Path.cwd()/"data"  # Change this if your CSV files are in another directory.
OUTPUT_DIR = Path.cwd()/"output"
LOOKBACK = 60
HIDDEN_SIZE = 50
DROPOUT = 0.2
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.001
MAX_GRAD_NORM = 1.0  # Clip the combined gradient norm before each Adam update.
SEED = 42

# Seeding makes runs more repeatable, but different devices/backends can still
# produce different floating-point results; exact reproducibility is not promised.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# CUDA is used on supported NVIDIA GPUs, MPS on supported Macs, otherwise CPU.
# All model parameters and input tensors must be on the same device.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"PyTorch {torch.__version__}; device: {device}")
# If your installed MPS backend cannot run an operation, choose device = torch.device("cpu").

## 2. Load and clean the time series
Only `Date` and `Open` are needed. Convert non-price rows (such as dividend annotations) to missing values and remove them, then sort chronologically. Select `Open` by name rather than by its column position. We do not need to export an Excel file to train the model.

In [ ]:
# Step 2: Find CSV files, clean prices, and verify a chronological train/test split.
def find_csv(*names):
    # Accept a folder written as either a string or a pathlib.Path object.
    data_dir = Path(DATA_DIR).expanduser()
    # Accept either filename spelling used in the original course materials.
    for name in names:
        path = data_dir / name
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"Expected one of {names} in {data_dir.resolve()}. "
        "Copy the original CSV files there or change DATA_DIR."
    )


def load_prices(path):
    # Keep only the two columns used by this forecasting exercise.
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.strip()
    if not {"Date", "Open"}.issubset(frame.columns):
        raise ValueError(f"{path.name} must contain Date and Open columns.")
    frame = frame[["Date", "Open"]].copy()
    # Parsing each entry supports both copied Yahoo dates and ISO dates.
    frame["Date"] = frame["Date"].map(lambda value: pd.to_datetime(value, errors="coerce"))
    frame["Open"] = pd.to_numeric(
        frame["Open"].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
    # Remove non-price rows, then order observations from oldest to newest.
    frame = frame.dropna(subset=["Date", "Open"])
    frame = frame.sort_values("Date").reset_index(drop=True)
    if frame.empty or not np.isfinite(frame["Open"].to_numpy()).all():
        raise ValueError(f"{path.name} contains no usable prices or contains infinite values.")
    if frame["Date"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate dates; resolve them before training.")
    return frame


mu_train = load_prices(find_csv("MU.csv"))
mu_test = load_prices(find_csv("MU_test.csv"))
if len(mu_train) <= LOOKBACK:
    raise ValueError(f"Training data needs more than {LOOKBACK} observations.")
if mu_train["Date"].max() >= mu_test["Date"].min():
    raise ValueError("All training dates must precede all test dates.")
print(f"Training rows: {len(mu_train)}; test rows: {len(mu_test)}")
print("Training dates:", mu_train["Date"].min(), "to", mu_train["Date"].max())
print("Test dates:", mu_test["Date"].min(), "to", mu_test["Date"].max())
mu_train.head()

In [ ]:
# Plot the cleaned training series to inspect its dates and price scale.
mu_train.plot(x="Date", y="Open", figsize=(10, 4), title="Micron (MU) training opening prices")
plt.ylabel("Opening price (USD)")
plt.show()

## 3. Scale prices and construct training windows
Fit the scaler **only on training prices**. Reuse this same scaler for the test data; future prices may legitimately scale outside `[0, 1]`.

For each target at position `i`, the input contains observations `i - 60` through `i - 1`. The target itself is excluded. With `N` training observations, there are `N - 60` examples.

PyTorch's RNN with `batch_first=True` takes `(batch, time, features)`. Here each example has shape `(60, 1)`. The last dimension counts input features, not labels.

In [ ]:
# Step 3: Fit scaling on training prices, then pair past windows with next prices.
mu_train_open = mu_train[["Open"]].to_numpy(dtype=np.float32)
mu_test_open = mu_test[["Open"]].to_numpy(dtype=np.float32)
scaler = MinMaxScaler(feature_range=(0, 1))
mu_train_scaled = scaler.fit_transform(mu_train_open).astype(np.float32)

# Slice all feature columns, preserving the single-feature dimension.
features_set = np.stack([
    mu_train_scaled[i - LOOKBACK:i]
    for i in range(LOOKBACK, len(mu_train_scaled))
])
# Target row i follows the 60 input observations and is excluded from that window.
labels = mu_train_scaled[LOOKBACK:].copy()
print("Inputs:", features_set.shape)  # (N - 60, 60, 1)
print("Targets:", labels.shape)       # (N - 60, 1)

## 4. Convert NumPy arrays into PyTorch batches
`TensorDataset` pairs each window with its target. `DataLoader` yields mini-batches. Shuffling complete training windows is valid here: the order **inside** each window is preserved and hidden states are reset for every forward pass. We never randomly mix training and test dates.

In [ ]:
# Step 4: Pair tensors and organize them into shuffled training mini-batches.
# float32 matches the default dtype of the model parameters. NumPy often uses
# float64, so explicitly converting prevents a common dtype mismatch.
X_train = torch.from_numpy(features_set).float()
y_train = torch.from_numpy(labels).float()
train_dataset = TensorDataset(X_train, y_train)

# A dedicated random generator makes the order of shuffled batches repeatable.
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    num_workers=0,  # Works directly in notebook kernels without worker setup.
)
# Tensors stay on CPU until each batch is moved to the chosen device.
print(f"{len(train_dataset)} examples in {len(train_loader)} batches per epoch")

## 5. Define a four-layer vanilla RNN
At each time step, a vanilla RNN updates its hidden state using the current input and the previous hidden state:

$$h_t = \tanh(W_{ih}x_t + b_{ih} + W_{hh}h_{t-1} + b_{hh}).$$

The same weights are reused for every time step. Unlike an LSTM, this RNN has **no gates and no separate cell state**. It can be harder to learn dependencies across long sequences because gradients may vanish or explode. Gradient clipping helps with exploding gradients; it does not solve vanishing gradients.

Each `nn.RNN` below has one recurrent layer (`num_layers=1`). Stacking four separate modules makes the dropout after each layer explicit. Layers two through four receive 50 features per time step from the preceding layer. The first three layers pass their entire sequence onward; we select the fourth layer's final time step to predict the next price.

| Operation | Shape |
|---|---|
| Input batch (`batch_first=True`) | `(B, 60, 1)` |
| Each RNN's sequence output | `(B, 60, 50)` |
| Final hidden state from each single-layer RNN | `(1, B, 50)` |
| Fourth layer's last sequence output | `(B, 50)` |
| Linear regression output | `(B, 1)` |

`nn.RNN` returns `(sequence_output, h_n)`, not the LSTM's `(sequence_output, (h_n, c_n))`. With this unidirectional, single-layer configuration, `sequence_output[:, -1, :]` equals `h_n[0]`. Hidden states default to zero on each forward call: batches and windows do not carry state into one another. `batch_first` changes the sequence layout, but does not change the hidden-state layout.

In [ ]:
class MuRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, dropout=0.2):
        # Initialize nn.Module so PyTorch tracks child layers and parameters.
        super().__init__()

        # Each module processes a full sequence using a tanh recurrence.
        # batch_first=True means inputs/outputs use (batch, time, features).
        # The first layer sees one price feature; later layers see hidden_size
        # learned features at each time step from the preceding RNN.
        self.rnn1 = nn.RNN(input_size, hidden_size, num_layers=1,
                           nonlinearity="tanh", batch_first=True)
        self.rnn2 = nn.RNN(hidden_size, hidden_size, num_layers=1,
                           nonlinearity="tanh", batch_first=True)
        self.rnn3 = nn.RNN(hidden_size, hidden_size, num_layers=1,
                           nonlinearity="tanh", batch_first=True)
        self.rnn4 = nn.RNN(hidden_size, hidden_size, num_layers=1,
                           nonlinearity="tanh", batch_first=True)

        # Explicit dropout randomly zeros 20% of activations during training.
        # model.eval() disables these layers' random masking for prediction.
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        self.dropout4 = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # Input x: (B, LOOKBACK, 1). No h_0 is supplied, so each RNN starts
        # with zeros for every example. There is no separate cell state.
        x, _ = self.rnn1(x)      # Sequence: (B, LOOKBACK, hidden_size).
        x = self.dropout1(x)    # Dropout keeps the same shape.
        x, _ = self.rnn2(x)      # '_' discards the final hidden-state tensor.
        x = self.dropout2(x)
        x, _ = self.rnn3(x)      # Keep all time steps for the next layer.
        x = self.dropout3(x)
        x, _ = self.rnn4(x)

        # Select the last TIME step for every example, not the last batch row.
        # This hidden vector encodes the window: (B, LOOKBACK, 50) -> (B, 50).
        x = x[:, -1, :]
        x = self.dropout4(x)
        # Predict one scaled price per window: (B, 50) -> (B, 1).
        # No sigmoid: future scaled prices can be outside the training range.
        return self.output(x)


# Store constructor settings so we can rebuild the model when loading weights.
model_config = {"input_size": 1, "hidden_size": HIDDEN_SIZE, "dropout": DROPOUT}
model = MuRNN(**model_config).to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Train with an explicit PyTorch loop
The loss is mean squared error on **scaled prices**. Adam updates the model parameters. Unlike Keras `fit`, PyTorch exposes each step: clear old gradients, predict, compute loss, backpropagate, and update weights.

Targets and predictions must both have shape `(B, 1)`. A target of shape `(B,)` can accidentally broadcast into a `(B, B)` error matrix. The assertion below guards against this.

We retain the original 100 epochs and batch size of 32. 

`loss.backward()` differentiates through all 60 time steps (backpropagation through time). We then clip the combined gradient norm to at most `MAX_GRAD_NORM` before `optimizer.step()`. Clipping rescales gradients if needed; it does not clip the loss or the parameter values.

In [ ]:
# Step 6: Train the RNN with MSE loss, backpropagation, clipping, and Adam.
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = {"loss": []}  # A plain dictionary replaces the Keras History object.

for epoch in range(EPOCHS):
    model.train()  # Enables dropout; this does not itself perform training.
    total_squared_error = 0.0
    examples_seen = 0

    for batch_x, batch_y in train_loader:
        # Inputs, targets, and model parameters must share a device.
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Gradients accumulate by default in PyTorch. Clear the previous batch's
        # gradients before calculating new ones; None can avoid unnecessary fills.
        optimizer.zero_grad(set_to_none=True)
        batch_predictions = model(batch_x)  # Calls forward and builds an autograd graph.
        assert batch_predictions.shape == batch_y.shape
        loss = criterion(batch_predictions, batch_y)
        loss.backward()   # Compute gradients d(loss)/d(parameter) using autograd.
        # Repeated recurrent updates can amplify gradients. Clip their global
        # norm after backpropagation and before the optimizer consumes them.
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()  # Apply Adam's parameter updates using clipped gradients.

        # item() detaches this scalar from autograd. Weight batch mean losses by
        # batch size so a small final batch does not count as a full-size batch.
        batch_count = batch_x.size(0)
        total_squared_error += loss.item() * batch_count
        examples_seen += batch_count

    epoch_loss = total_squared_error / examples_seen
    history["loss"].append(epoch_loss)
    if epoch == 0 or (epoch + 1) % 10 == 0 or epoch + 1 == EPOCHS:
        print(f"Epoch {epoch + 1:3d}/{EPOCHS}: training MSE = {epoch_loss:.6f}")

In [ ]:
# Plot the recorded training loss; this is not a validation or test error curve.
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history["loss"]) + 1), history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE (scaled prices)")
plt.title("Training loss")
plt.legend()
plt.show()

## 7. Prepare chronological test windows
This is **rolling one-step-ahead evaluation**. The first test prediction uses the last 60 training observations. Each later prediction uses actual opening prices observed before that test day, including earlier test observations. It is not a recursive forecast of the entire test period made on the last training day.

Use the training scaler without refitting it. Window lengths are derived from the CSV contents, replacing the original hard-coded 1,975 training rows and 37 test rows.

In [ ]:
# Step 7: Attach the final training window to observed test prices for context.
context_prices = np.concatenate([mu_train_open[-LOOKBACK:], mu_test_open], axis=0)
context_scaled = scaler.transform(context_prices).astype(np.float32)
test_features = np.stack([
    context_scaled[i:i + LOOKBACK]
    for i in range(len(mu_test_open))
])
# Window i stops immediately before target i; it never includes that day's price.
X_test = torch.from_numpy(test_features).float()
test_loader = DataLoader(TensorDataset(X_test), batch_size=BATCH_SIZE, shuffle=False)
print("Test inputs:", X_test.shape)  # (number of test observations, 60, 1)

## 8. Predict and convert back to dollars
`model.eval()` disables dropout. It does **not** disable gradient tracking; `torch.inference_mode()` handles that separately. Move predictions back to CPU before converting to NumPy for scikit-learn and plotting.

In [ ]:
# Step 8: Predict in chronological order with dropout and gradient tracking disabled.
model.eval()
prediction_batches = []
with torch.inference_mode():
    for (batch_x,) in test_loader:
        batch_predictions = model(batch_x.to(device))
        prediction_batches.append(batch_predictions.cpu().numpy())

scaled_predictions = np.concatenate(prediction_batches, axis=0)
# Undo the training scale so predictions and errors are expressed in dollars.
predictions = scaler.inverse_transform(scaled_predictions)
assert predictions.shape == mu_test_open.shape

# A persistence baseline predicts that the next opening price equals the last
# observed opening price. Compare against it rather than judging only a plot.
baseline = context_prices[LOOKBACK - 1:-1]
for name, values in [("RNN", predictions), ("Previous-open baseline", baseline)]:
    errors = values - mu_test_open
    print(f"{name}: MAE = ${np.abs(errors).mean():.2f}; RMSE = ${np.sqrt(np.mean(errors ** 2)):.2f}")

In [ ]:
# Compare predictions, actual prices, and the simple baseline against real dates.
plt.figure(figsize=(11, 5))
plt.plot(mu_test["Date"], mu_test_open[:, 0], color="blue", label="Actual opening price")
plt.plot(mu_test["Date"], predictions[:, 0], color="red", label="RNN prediction")
plt.plot(mu_test["Date"], baseline[:, 0], color="gray", linestyle="--", label="Previous-open baseline")
plt.title("Micron (MU) opening prices: rolling one-step-ahead predictions")
plt.xlabel("Date")
plt.ylabel("Opening price (USD)")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 9. Save and reload the PyTorch model
A `state_dict` stores learned parameter tensors. Reconstruct the same architecture before loading it. The checkpoint also stores the lookback, model configuration, training history, and fitted scaler parameters, which are needed to interpret predictions consistently.

The checkpoint below supports inference; resuming Adam training exactly would additionally require optimizer and random-generator states. Load only checkpoints you trust.

In [ ]:
# Step 9: Save learned weights and preprocessing settings to an RNN-specific file.
# Normalize the output folder too, in case it was entered as a string.
output_dir = Path(OUTPUT_DIR).expanduser()
checkpoint_path = output_dir / "mu_rnn_pytorch.pt"
checkpoint = {
    # Save CPU copies for portability across CPU, CUDA, and MPS environments.
    "model_state_dict": {key: value.detach().cpu().clone() for key, value in model.state_dict().items()},
    "model_config": model_config,
    "lookback": LOOKBACK,
    "model_type": "vanilla_rnn_tanh",
    "max_grad_norm": MAX_GRAD_NORM,
    "history": history,
    # Lists and primitive values can be loaded with weights_only=True.
    "scaler": {
        "feature_range": list(scaler.feature_range),
        "scale_": scaler.scale_.tolist(),
        "min_": scaler.min_.tolist(),
        "data_min_": scaler.data_min_.tolist(),
        "data_max_": scaler.data_max_.tolist(),
        "data_range_": scaler.data_range_.tolist(),
        "n_features_in_": int(scaler.n_features_in_),
        "n_samples_seen_": int(scaler.n_samples_seen_),
    },
}
output_dir.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, checkpoint_path)
print("Saved:", checkpoint_path.resolve())

In [ ]:
# Reconstruct the RNN and scaler, then verify the checkpoint round trip.
# Load tensors on CPU first, then move the reconstructed model to your device.
saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
reloaded_model = MuRNN(**saved["model_config"])
reloaded_model.load_state_dict(saved["model_state_dict"])
reloaded_model = reloaded_model.to(device)
reloaded_model.eval()  # New modules start in training mode, so disable dropout again.

reloaded_scaler = MinMaxScaler(feature_range=tuple(saved["scaler"]["feature_range"]))
for name in ("scale_", "min_", "data_min_", "data_max_", "data_range_"):
    setattr(reloaded_scaler, name, np.asarray(saved["scaler"][name], dtype=np.float32))
reloaded_scaler.n_features_in_ = saved["scaler"]["n_features_in_"]
reloaded_scaler.n_samples_seen_ = saved["scaler"]["n_samples_seen_"]

# Verify that saving/loading preserved predictions for the same input batch.
with torch.inference_mode():
    sample = X_test[:BATCH_SIZE].to(device)
    expected = model(sample)
    actual = reloaded_model(sample)
    torch.testing.assert_close(actual, expected)
np.testing.assert_allclose(
    reloaded_scaler.inverse_transform(scaled_predictions), predictions
)
print("Reloaded model and scaler reproduce the original predictions.")